In [2]:
from pathlib import Path
import pandas as pd
import numpy as np


# %pip install openpyxl


In [3]:
DATA_DIR = Path("C:\\Users\\Home\\Documents\\Datos Ebsa")
OUT_DIR = Path("C:\\Users\\Home\\Documents\\Datos Ebsa\\Procesado")

OUT_DIR.mkdir(exist_ok=True)

archivos = sorted(DATA_DIR.glob("*.xlsx"))

print(f"Archivos encontrados: {len(archivos)}")

for archivo in archivos[:10]:
    print(archivo.name)

Archivos encontrados: 12
formato_tc2_20251.xlsx
formato_tc2_202510.xlsx
formato_tc2_202511.xlsx
formato_tc2_202512.xlsx
formato_tc2_20252.xlsx
formato_tc2_20253.xlsx
formato_tc2_20254.xlsx
formato_tc2_20255.xlsx
formato_tc2_20256.xlsx
formato_tc2_20257.xlsx


In [4]:
COLUMNAS = [
    "NIU",
    "ID Factura",
    "Tipo Factura",
    "Año de reporte",
    "Mes de reporte",
    "Días Facturados",
    "Consumo Usuario (kWh)",
    "Refacturación por Consumo Usuario - (kWh)",
    "Fecha de Lectura Actual",
    "Fecha de Lectura Anterior",
    "Tipo de Lectura",
    "Ciclo",
    "Clase de Servicio"
]

In [5]:
def procesar_archivo(ruta):
    
    print(f"Procesando: {ruta.name}")
    
    df = pd.read_excel(
        ruta,
        usecols=lambda col: col in COLUMNAS,
        engine="openpyxl"
    )

    # -------------------------
    # NIU
    # -------------------------
    
    df["NIU"] = (
        df["NIU"]
        .astype("string")
        .str.strip()
    )

    # -------------------------
    # Variables numéricas
    # -------------------------
    
    numericas = [
        "Año de reporte",
        "Mes de reporte",
        "Días Facturados",
        "Consumo Usuario (kWh)",
        "Refacturación por Consumo Usuario - (kWh)",
        "Ciclo"
    ]

    for col in numericas:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # -------------------------
    # Fechas
    # -------------------------

    df["Fecha de Lectura Actual"] = pd.to_datetime(
        df["Fecha de Lectura Actual"],
        errors="coerce",
        dayfirst=True
    )

    df["Fecha de Lectura Anterior"] = pd.to_datetime(
        df["Fecha de Lectura Anterior"],
        errors="coerce",
        dayfirst=True
    )

    # -------------------------
    # Eliminar registros sin
    # información básica
    # -------------------------

    df = df.dropna(
        subset=[
            "NIU",
            "Año de reporte",
            "Mes de reporte"
        ]
    )

    df["Año de reporte"] = df["Año de reporte"].astype(int)
    df["Mes de reporte"] = df["Mes de reporte"].astype(int)

    # -------------------------
    # Crear fecha mensual
    # -------------------------

    df["periodo"] = pd.to_datetime(
        dict(
            year=df["Año de reporte"],
            month=df["Mes de reporte"],
            day=1
        )
    )

    return df

In [6]:
df_prueba = procesar_archivo(archivos[0])

df_prueba.head()

Procesando: formato_tc2_20251.xlsx


,NIU,Tipo Factura,ID Factura,Fecha de Lectura Actual,Fecha de Lectura Anterior,Días Facturados,Tipo de Lectura,Consumo Usuario (kWh),Refacturación por Consumo Usuario - (kWh),Año de reporte,Mes de reporte,Ciclo,Clase de Servicio,periodo
0,401172249,1,209445439,2025-01-09,2024-12-20,20,1,1731,-280,2025,1,0,CR,2025-01-01
1,401172249,6,207802006,2025-01-09,2024-12-20,20,1,0,-280,2025,1,0,0,2025-01-01
2,409663204,1,209497498,2025-01-09,2024-12-20,20,1,908,0,2025,1,0,CR,2025-01-01
3,417077568,1,209499649,2025-01-09,2024-12-20,20,1,24,0,2025,1,0,RS,2025-01-01
4,437958264,1,209475874,2025-01-09,2024-12-20,20,1,127,0,2025,1,0,RS,2025-01-01


In [7]:

df_prueba.shape

(564705, 14)

In [8]:
df_prueba.dtypes

NIU                                                  string
Tipo Factura                                          int64
ID Factura                                            int64
Fecha de Lectura Actual                      datetime64[us]
Fecha de Lectura Anterior                    datetime64[us]
Días Facturados                                       int64
Tipo de Lectura                                       int64
Consumo Usuario (kWh)                                 int64
Refacturación por Consumo Usuario - (kWh)             int64
Año de reporte                                        int64
Mes de reporte                                        int64
Ciclo                                                 int64
Clase de Servicio                                       str
periodo                                      datetime64[us]
dtype: object

In [9]:
duplicados = (
    df_prueba
    .groupby(["NIU", "periodo"])
    .size()
    .reset_index(name="cantidad_registros")
)

duplicados[
    duplicados["cantidad_registros"] > 1
].sort_values(
    "cantidad_registros",
    ascending=False
).head(20)

,NIU,periodo,cantidad_registros
415823,576040024,2025-01-01,36
17710,104133490,2025-01-01,30
349407,462891380,2025-01-01,27
405556,558225939,2025-01-01,21
476510,847254080,2025-01-01,21
108689,1216853058,2025-01-01,21
326026,426083207,2025-01-01,20
445239,704241986,2025-01-01,20
171074,146251427,2025-01-01,18
337001,442954408,2025-01-01,16


In [10]:
resumen_mes = (
    df_prueba
    .groupby(
        ["NIU", "periodo"],
        as_index=False
    )
    .agg(
        cantidad_registros=("ID Factura", "size"),

        dias_facturados_max=(
            "Días Facturados",
            "max"
        ),

        dias_facturados_mediana=(
            "Días Facturados",
            "median"
        ),

        fecha_lectura_actual=(
            "Fecha de Lectura Actual",
            "max"
        ),

        fecha_lectura_anterior=(
            "Fecha de Lectura Anterior",
            "min"
        ),

        ciclo=(
            "Ciclo",
            "first"
        ),

        ciclos_diferentes=(
            "Ciclo",
            "nunique"
        ),

        tipo_lectura=(
            "Tipo de Lectura",
            "first"
        ),

        clase_servicio=(
            "Clase de Servicio",
            "first"
        )
    )
)

In [11]:
resumen_mes.head()

,NIU,periodo,cantidad_registros,dias_facturados_max,dias_facturados_mediana,fecha_lectura_actual,fecha_lectura_anterior,ciclo,ciclos_diferentes,tipo_lectura,clase_servicio
0,1000006589,2025-01-01,1,31,31.0,2025-01-18,2024-12-18,1,1,1,RS
1,1000007366,2025-01-01,1,32,32.0,2025-01-22,2024-12-21,1,1,1,RS
2,1000008143,2025-01-01,1,34,34.0,2025-01-23,2024-12-20,1,1,1,RS
3,100000949,2025-01-01,1,93,93.0,2025-01-11,2024-10-10,12,1,1,RS
4,1000009920,2025-01-01,1,34,34.0,2025-01-23,2024-12-20,1,1,1,RS


In [ ]:
resumenes = []

for archivo in archivos:
    
    df = procesar_archivo(archivo)

    resumen = (
        df
        .groupby(
            ["NIU", "periodo"],
            as_index=False
        )
        .agg(
            cantidad_registros=("ID Factura", "size"),

            dias_facturados_max=(
                "Días Facturados",
                "max"
            ),

            dias_facturados_mediana=(
                "Días Facturados",
                "median"
            ),

            fecha_lectura_actual=(
                "Fecha de Lectura Actual",
                "max"
            ),

            fecha_lectura_anterior=(
                "Fecha de Lectura Anterior",
                "min"
            ),

            ciclo=(
                "Ciclo",
                "first"
            ),

            ciclos_diferentes=(
                "Ciclo",
                "nunique"
            ),

            tipo_lectura=(
                "Tipo de Lectura",
                "first"
            ),

            clase_servicio=(
                "Clase de Servicio",
                "first"
            )
        )
    )

    resumenes.append(resumen)

    # Liberar memoria
    del df

Procesando: formato_tc2_20251.xlsx
Procesando: formato_tc2_202510.xlsx
Procesando: formato_tc2_202511.xlsx
Procesando: formato_tc2_202512.xlsx
Procesando: formato_tc2_20252.xlsx
Procesando: formato_tc2_20253.xlsx
Procesando: formato_tc2_20254.xlsx
Procesando: formato_tc2_20255.xlsx
Procesando: formato_tc2_20256.xlsx
Procesando: formato_tc2_20257.xlsx
Procesando: formato_tc2_20258.xlsx


In [ ]:
historico = pd.concat(
    resumenes,
    ignore_index=True
)

In [ ]:
historico.shape

In [ ]:
historico.to_parquet(
    OUT_DIR / "historico_niu_mes_2022_2025.parquet",
    index=False
)

In [ ]:
historico["presente"] = 1

In [ ]:
matriz_presencia = (
    historico
    .pivot_table(
        index="NIU",
        columns="periodo",
        values="presente",
        aggfunc="max",
        fill_value=0
    )
    .astype("uint8")
)

In [ ]:
meses = pd.date_range(
    start="2022-01-01",
    end="2025-12-01",
    freq="MS"
)

In [ ]:
matriz_presencia = matriz_presencia.reindex(
    columns=meses,
    fill_value=0
)

In [ ]:
matriz_presencia.shape